<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

<div style="text-align: center;">

# المپیاد هوش مصنوعی | آزمون عملی جامع | تابستان ۱۴۰۵

</div>

**به نام خدا**

- مدت آزمون: **۲۴۰ دقیقه**
- مجموع نمره: **۱۵۰**
- این آزمون شش مسئلهٔ مستقل دارد؛ می‌توانید به هر ترتیبی که می‌خواهید حل کنید.
- در سلول‌های `# TODO` کد خود را بنویسید. سلول‌های علامت‌گذاری‌شده با «فقط اجرا کنید» (read-only) را تغییر ندهید.
- فقط از `numpy`، `matplotlib` و `torch` استفاده کنید (و `sklearn` صرفاً در جایی که صریحاً مجاز اعلام شده است).
- برای نمرهٔ کامل، کد باید بدون خطا اجرا شود و نتایج خواسته‌شده را تولید کند. توضیح کوتاه فارسی در انتهای هر مسئله دربارهٔ مشاهده‌هایتان بنویسید.
- ارزیابی هر بخش در انتهای همان بخش با `assert` یا سلول بررسی انجام می‌شود؛ مطمئن شوید همهٔ سلول‌ها بدون خطا (بدون `AssertionError`) اجرا می‌شوند.

</div>

<style>
.katex, .katex-display, .katex-html, .katex-mathml { direction: ltr !important; unicode-bidi: isolate; }
div[dir="ltr"] { text-align: center; }
</style>


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

## ۰. آماده‌سازی

سلول زیر را اجرا کنید:

</div>

In [1]:
# آماده‌سازی — فقط اجرا کنید، تغییر ندهید
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 1405
np.random.seed(SEED)
torch.manual_seed(SEED)
plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['figure.dpi'] = 100
print("numpy:", np.__version__, "| torch:", torch.__version__)

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

## مسئلهٔ ۱ — رگرسیون لجستیک با regularization L2 از صفر (۲۵ نمره)

می‌خواهیم رگرسیون لجستیک را **بدون استفاده از کتابخانه‌های یادگیری ماشین** پیاده کنیم. مدل به‌ازای ورودی ⁦$x \in \mathbb{R}^2$⁩ و پارامترهای ⁦$w \in \mathbb{R}^2$⁩، ⁦$b \in \mathbb{R}$⁩:

<div dir="ltr">

$$
\hat{p}(x) = \sigma(w^\top x + b), \qquad \sigma(z) = \frac{1}{1+e^{-z}}
$$

</div>

تابع هزینه با regularization L2 (بدون رگولاریز کردن بایاس):

<div dir="ltr">

$$
J(w, b) = -\frac{1}{n}\sum_{i=1}^{n}\Big[y_i \ln \hat{p}(x_i) + (1-y_i)\ln(1-\hat{p}(x_i))\Big] + \frac{\lambda}{2n}\|w\|^2
$$

</div>

دادهٔ مسئله (دو خوشهٔ گاوسی هم‌پوشان) در سلول زیر ساخته می‌شود.

</div>

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

</div>

In [2]:
# ساخت داده — فقط اجرا کنید، تغییر ندهید
rng = np.random.default_rng(SEED)
n_per_class = 300
X1 = rng.normal(loc=[2.0, 2.0], scale=1.1, size=(n_per_class, 2))
X0 = rng.normal(loc=[0.0, 0.0], scale=1.1, size=(n_per_class, 2))
X_t1 = np.vstack([X1, X0]).astype(np.float64)          # (600, 2)
y_t1 = np.hstack([np.ones(n_per_class), np.zeros(n_per_class)]).astype(np.float64)

# تقسیم آموزش/آزمون
idx = rng.permutation(len(X_t1))
n_tr = 400
tr, te = idx[:n_tr], idx[n_tr:]
X1_tr, y1_tr, X1_te, y1_te = X_t1[tr], y_t1[tr], X_t1[te], y_t1[te]

plt.scatter(X1_tr[y1_tr==1][:,0], X1_tr[y1_tr==1][:,1], s=12, alpha=0.6, label='کلاس ۱')
plt.scatter(X1_tr[y1_tr==0][:,0], X1_tr[y1_tr==0][:,1], s=12, alpha=0.6, label='کلاس ۰')
plt.legend(); plt.title('دادهٔ آموزش مسئلهٔ ۱'); plt.gca().set_aspect('equal')
plt.show()

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۱-الف) (۶ نمره)** توابع `sigmoid(z)`، `cost(w, b, X, y, lam)` و `grads(w, b, X, y, lam)` را بنویسید. `cost` باید هزینهٔ ⁦$J$⁩ بالا و `grads` باید ⁦$\frac{\partial J}{\partial w}$⁩ و ⁦$\frac{\partial J}{\partial b}$⁩ را (به‌صورت دقیق، برداری) برگرداند. برای پایداری عددی از np.clip روی ورودی لگاریتم استفاده کنید (مثلاً در بازهٔ ⁦$[10^{-12}, 1-10^{-12}]$⁩).

**توجه:** گرادیان‌ها را خودتان به‌صورت تحلیلی استخراج کنید؛ درستی‌شان را با تفاضل متناهی در سلول بررسی تأیید کنید (سلول بررسی با مشتق عددی شما مقایسه می‌کند، نه با فرمول بسته).

</div>

In [ ]:
def sigmoid(z):
    # TODO: پیاده‌سازی سیگموئید به‌صورت برداری
    pass

def cost(w, b, X, y, lam=0.0):
    # TODO: هزینهٔ آنتروپی متقاطع + رگولاریزاسیون L2 (بدون بایاس)
    pass

def grads(w, b, X, y, lam=0.0):
    # TODO: بازگرداندن (grad_w, grad_b)
    pass


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

</div>

In [ ]:
# بررسی تفاضل متناهی — فقط اجرا کنید؛ بدون خطا باید اجرا شود
w0 = np.array([0.3, -0.7]); b0 = 0.2; lam0 = 0.5
gw, gb = grads(w0, b0, X1_tr, y1_tr, lam0)
eps = 1e-6
for j in range(2):
    e = np.zeros(2); e[j] = eps
    num = (cost(w0 + e, b0, X1_tr, y1_tr, lam0) - cost(w0 - e, b0, X1_tr, y1_tr, lam0)) / (2 * eps)
    assert abs(num - gw[j]) < 1e-4, f"gradient mismatch in w[{j}]: {num} vs {gw[j]}"
num_b = (cost(w0, b0 + eps, X1_tr, y1_tr, lam0) - cost(w0, b0 - eps, X1_tr, y1_tr, lam0)) / (2 * eps)
assert abs(num_b - gb) < 1e-4, f"gradient mismatch in b: {num_b} vs {gb}"
print("همهٔ گرادیان‌ها با تفاضل متناهی سازگارند ✓")

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۱-ب) (۷ نمره)** حلقهٔ آموزش `train_logreg(X, y, lam, lr, epochs)` را بنویسید: با ⁦$w=0$⁩، ⁦$b=0$⁩ شروع کنید و با gradient descent تمام‌دسته‌ای (full-batch) پارامترها را به‌روزرسانی کنید؛ تاریخچهٔ هزینه را برگردانید. سپس مدل را با ⁦$\lambda = 0$⁩ آموزش دهید، منحنی هزینه را رسم کنید و **دقت (accuracy)** روی دادهٔ آموزش و آزمون را گزارش کنید (آستانهٔ تصمیم ⁦$0.5$⁩).

</div>

In [ ]:
def train_logreg(X, y, lam=0.0, lr=0.1, epochs=500):
    # TODO: آموزش با گرادیان کاهشی؛ بازگرداندن w, b, history (لیست هزینه در هر epoch)
    pass

w1, b1, hist1 = train_logreg(X1_tr, y1_tr, lam=0.0)
# TODO: رسم منحنی هزینه + گزارش دقت آموزش و آزمون


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۱-ج) (۶ نمره)** اثر regularization: برای ⁦$\lambda \in \{0, 0.01, 0.1, 1, 10, 100\}$⁩ مدل را آموزش دهید و نمودار دقت آزمون بر حسب ⁦$\lambda$⁩ (محور لگاریتمی) را رسم کنید. کدام ⁦$\lambda$⁩ بهتر است؟ چه الگویی می‌بینید و چرا؟

</div>

In [ ]:
lams = [0, 0.01, 0.1, 1, 10, 100]
# TODO: آموزش برای هر lambda، رسم دقت آزمون بر حسب lambda و تحلیل


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۱-د) (۶ نمره)** مرز تصمیم: مدل با بهترین ⁦$\lambda$⁩ را روی شبکهٔ نقاط ⁦$[-4,6]\times[-4,6]$⁩ ارزیابی کنید و مرز تصمیم ⁦$\sigma(w^\top x + b) = 0.5$⁩ را همراه با نقاط آزمون رسم کنید. سپس نشان دهید مرز تصمیم رگرسیون لجستیک همیشه **خطی** است؛ یعنی معادلهٔ صریح خط ⁦$w^\top x + b = 0$⁩ را برای مدل خودتان بنویسید و با رسم بررسی کنید که contour سطح ⁦$0.5$⁩ روی همین خط است.

</div>

In [ ]:
# TODO: رسم مرز تصمیم (contour سطح 0.5) + نقاط آزمون + معادلهٔ صریح خط


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**توضیح کوتاه (لازم برای نمره):** مشاهدات خود دربارهٔ روند هزینه، اثر ⁦$\lambda$⁩ و خطی بودن مرز تصمیم در یک سلول markdown بنویسید.

</div>

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

## مسئلهٔ ۲ — درخت تصمیم و bagging از صفر (۲۵ نمره)

درخت تصمیم رگرسیون/طبقه‌بندی را با معیار **Gini** پیاده می‌کنیم و سپس با **bagging** (Bootstrap AGGregatING) جمعش می‌کنیم. دادهٔ این مسئله یک مسئلهٔ طبقه‌بندی دوبعدی غیرخطی (چهار خوشهٔ گوشه‌ای) است.

</div>

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

</div>

In [8]:
# ساخت داده — فقط اجرا کنید، تغییر ندهید
rng = np.random.default_rng(SEED + 1)
n_pc = 150
centers = np.array([[1.5, 1.5], [-1.5, -1.5], [1.5, -1.5], [-1.5, 1.5]])
labels_of_center = [1, 1, 0, 0]   # XOR-like
Xs, ys = [], []
for c, lab in zip(centers, labels_of_center):
    Xs.append(rng.normal(loc=c, scale=0.55, size=(n_pc, 2)))
    ys.append(np.full(n_pc, lab))
X_t2 = np.vstack(Xs); y_t2 = np.hstack(ys)
idx = rng.permutation(len(X_t2)); n_tr = 400
tr2, te2 = idx[:n_tr], idx[n_tr:]
X2_tr, y2_tr, X2_te, y2_te = X_t2[tr2], y_t2[tr2], X_t2[te2], y_t2[te2]

def plot_t2(title):
    plt.scatter(X2_tr[y2_tr==1][:,0], X2_tr[y2_tr==1][:,1], s=12, alpha=0.6, label='کلاس ۱ (آموزش)')
    plt.scatter(X2_tr[y2_tr==0][:,0], X2_tr[y2_tr==0][:,1], s=12, alpha=0.6, label='کلاس ۰ (آموزش)')
    plt.legend(); plt.title(title); plt.gca().set_aspect('equal')
plot_t2('دادهٔ مسئلهٔ ۲ (الگوی XOR)'); plt.show()

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۲-الف) (۷ نمره)** تابع `gini(y)` را بنویسید که ناخالصی Gini برچسب‌های `y` را حساب کند: ⁦$G(y) = 1 - \sum_c p_c^2$⁩. سپس تابع `best_split(X, y)` را بنویسید که از میان همهٔ (ویژگی، آستانه)های ممکن، شکافی را بیابد که **کاهش وزنی ناخالصی** را بیشینه کند. برای آستانه‌ها کافی است میانگین جفت‌های متوالی مقادیر یکتای هر ویژگی را امتحان کنید. خروجی: `(feature, threshold, gain)`.

</div>

In [ ]:
def gini(y):
    # TODO
    pass

def best_split(X, y):
    # TODO: خروجی (feature, threshold, gain)
    pass


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۲-ب) (۸ نمره)** کلاس `DecisionTree` را بنویسید که درخت را بازگشتی تا عمق `max_depth` بسازد. هر گره داخلی با `best_split` شکاف می‌خورد؛ برگ مقدار پیش‌بینی = کلاس اکثریت آن برگ را دارد. متدهای `fit(X, y)` و `predict(X)` لازم است. درخت را با `max_depth` های مختلف (۱، ۲، ۵، ۱۰) آموزش دهید و دقت آموزش/آزمون هرکدام را گزارش کنید. (توقع: عمق ۱ خطا می‌کند چون داده XOR است!)

</div>

In [ ]:
class DecisionTree:
    def __init__(self, max_depth=2):
        # TODO
        pass
    def fit(self, X, y):
        # TODO: ساخت بازگشتی درخت
        pass
    def predict(self, X):
        # TODO: پیمایش درخت برای هر نمونه
        pass


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۲-ج) (۶ نمره)** bagging: کلاس `BaggedTrees` را بنویسید که `n_estimators` درخت (پیش‌فرض ۲۵، `max_depth=5`) روی bootstrap-sampleهای دادهٔ آموزش می‌سازد و با رأی اکثریت پیش‌بینی می‌کند. سپس دقت آزمون bagging را با بهترین تک‌درخت مقایسه کنید و مرز تصمیم bagging را رسم کنید.

</div>

In [ ]:
class BaggedTrees:
    def __init__(self, n_estimators=25, max_depth=5):
        # TODO
        pass
    def fit(self, X, y):
        # TODO: برای هر درخت، نمونه‌گیری bootstrap با جای‌گذاری
        pass
    def predict(self, X):
        # TODO: رأی اکثریت
        pass


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۲-د) (۴ نمره)** در یک سلول markdown توضیح دهید: چرا تک‌درختِ عمیق روی این داده واریانس بالایی دارد و bagging چگونه این واریانس را کم می‌کند؟ حدس بزنید (و با یک آزمایش کوتاه عددی بررسی کنید) اگر به‌جای نمونه‌برداری bootstrap از کل داده برای همهٔ درخت‌ها استفاده کنیم چه اتفاقی می‌افتد؟

</div>

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

## مسئلهٔ ۳ — PCA و خوشه‌بندی k-means از صفر (۲۵ نمره)

در این مسئله دادهٔ گل‌مانند سه‌بعدی (خوشه‌های کشیده در جهات همبسته) می‌سازیم؛ ابتدا PCA را با تجزیه ویژه پیاده می‌کنیم و سپس k-means را با مقداردهی k-means++.

</div>

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

</div>

In [13]:
# ساخت داده — فقط اجرا کنید، تغییر ندهید
rng = np.random.default_rng(SEED + 2)
n = 600
# دادهٔ سه‌بعدی با ساختار رتبهٔ ~۲ (روی یک صفحهٔ کج در R^3)
z = rng.normal(size=(n, 2))
basis = np.array([[1.0, 0.2, 0.5],
                  [0.1, 1.0, -0.3]])          # دو جهت اصلی
X_t3 = z @ basis + np.array([0.5, -0.5, 1.0]) # جابه‌جایی
X_t3 += rng.normal(scale=0.05, size=X_t3.shape)  # نویز کوچک

fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(121, projection='3d'); ax.scatter(*X_t3.T, s=6, alpha=0.5); ax.set_title('دادهٔ مسئلهٔ ۳')
plt.show()

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۳-الف) (۸ نمره)** تابع `pca(X, k)` را بنویسید: میانگین را کم کنید، ماتریس کوواریانس تجربی ⁦$\Sigma = \frac{1}{n-1}(X-\mu)^\top(X-\mu)$⁩ را بسازید، با `np.linalg.eigh` بردارهای ویژه را بگیرید و ⁦$k$⁩ principal components (مرتب‌شده نزولی) و explained variance ratio را برگردانید. سپس:
1. explained variance ratioٔ مؤلفه‌های ۱ تا ۳ را چاپ کنید؛
2. داده را با ⁦$k=2$⁩ روی دو مؤلفهٔ اول تصویر و پراکنش‌شان را رسم کنید؛
3. **خطای بازسازی نسبی** ⁦$\frac{\|X - \hat{X}\|_F^2}{\|X\|_F^2}$⁩ را برای ⁦$k=1,2,3$⁩ حساب و چاپ کنید (برای بازسازی، ⁦$\mu$⁩ را برگردانید) و رابطه‌اش را با نسبت واریانس باقی‌مانده مقایسه کنید.

</div>

In [ ]:
def pca(X, k):
    # TODO: بازگرداندن (components (k,d), explained_ratio (k,), mean (d,))
    pass

def reconstruct(X, k):
    # TODO: بازسازی داده با k مؤلفه
    pass


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۳-ب) (۸ نمره)** k-means از صفر: توابع `kmeans_pp_init(X, k, rng)` و `kmeans(X, k, max_iter=100)` را بنویسید. الگوریتم تا همگرایی (عدم تغییر انتساب‌ها) یا `max_iter` تکرار می‌شود و `(labels, centers, inertia)` را برمی‌گرداند که `inertia` مجموع مربع فاصلهٔ نقاط تا مرکز خوشه‌شان است. روی دادهٔ مسئلهٔ ۱ (`X_t1`، دو خوشهٔ هم‌پوشان) با ⁦$k=2$⁩ اجرا کنید، خوشه‌ها را رسم کنید و inertia را گزارش کنید.

**توجه:** الگوریتم k-means++ برای مقداردهی اولیه: مرکز اول تصادفی؛ هر مرکز بعدی از میان نقاط داده با احتمال متناسب با مجذور فاصلهٔ تا نزدیک‌ترین مرکز موجود انتخاب می‌شود.

</div>

In [ ]:
def kmeans_pp_init(X, k, rng):
    # TODO: مقداردهی k-means++
    pass

def kmeans(X, k, max_iter=100):
    # TODO: خروجی (labels, centers, inertia)
    pass


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۳-ج) (۵ نمره)** با اجرای k-means با ۱۰ مقداردهی اولیهٔ متفاوت (seedهای مختلف)، نمودار مقادیر inertia را رسم کنید. آیا همیشه به سراسری‌ترین کمینه می‌رسد؟ سپس با روش «آرنج» (elbow) برای ⁦$k = 1,\dots,6$⁩ روی دادهٔ مسئلهٔ ۱، inertia بر حسب ⁦$k$⁩ را رسم کنید و بگویید چه ⁦$k$⁩ای طبیعی به نظر می‌رسد.

</div>

In [ ]:
# TODO: اجرای چندباره + نمودار elbow + تحلیل


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۳-د) (۴ نمره)** پیوند PCA و k-means: دادهٔ مسئلهٔ ۳ را با PCA به ⁦$k=2$⁩ بُعد ببرید، k-means را روی دادهٔ کاهش‌بُعد‌یافته اجرا کنید و نتیجه را با اجرای k-means روی دادهٔ اصلی سه‌بعدی مقایسه کنید (inertia را در فضای اصلی برای هر دو محاسبه کنید). آیا کاهش بُعد به هزینهٔ خوشه‌بندی افت کرد؟ چرا؟

</div>

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

## مسئلهٔ ۴ — MLP و بهینه‌ساز Adam از صفر (۲۵ نمره)

شبکهٔ عصبی دولایه را **کاملاً از صفر با NumPy** می‌سازیم: backpropagation دستی و بهینه‌ساز Adam. دادهٔ مسئله، دو مارپیچ (spirals) است که با هیچ مدل خطی جدا نمی‌شود.

</div>

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

</div>

In [18]:
# ساخت داده — فقط اجرا کنید، تغییر ندهید
rng = np.random.default_rng(SEED + 4)
n_per = 250
def make_spiral(n, delta, rng):
    r = np.sqrt(rng.random(n)) * 3.0
    t = 4.0 * np.pi * r / 3.0 + delta
    X = np.c_[r * np.sin(t), r * np.cos(t)] + rng.normal(0, 0.12, (n, 2))
    return X
X_t4 = np.vstack([make_spiral(n_per, 0, rng), make_spiral(n_per, np.pi, rng)])
y_t4 = np.hstack([np.zeros(n_per), np.ones(n_per)]).astype(int)
idx = rng.permutation(len(X_t4)); n_tr = 400
tr4, te4 = idx[:n_tr], idx[n_tr:]
X4_tr, y4_tr, X4_te, y4_te = X_t4[tr4], y_t4[tr4], X_t4[te4], y_t4[te4]

plt.scatter(X_t4[y_t4==0][:,0], X_t4[y_t4==0][:,1], s=10, alpha=0.6, label='کلاس ۰')
plt.scatter(X_t4[y_t4==1][:,0], X_t4[y_t4==1][:,1], s=10, alpha=0.6, label='کلاس ۱')
plt.legend(); plt.gca().set_aspect('equal'); plt.title('دادهٔ مارپیچی مسئلهٔ ۴'); plt.show()

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۴-الف) (۱۰ نمره)** کلاس `MLP` را با معماری ⁦$2 \to 64 \to 64 \to 2$⁩ (خروجی logits برای دو کلاس) و فعال‌سازی ReLU بنویسید. متدها:
- `forward(X)`: خروجی logits شکل ⁦$(n, 2)$⁩؛ مقادیر میانی را برای backpropagation ذخیره کند؛
- `backward(X, y_onehot)`: backpropagation از هزینهٔ softmax + cross-entropy (گرادیان مشترک ⁦$\hat{p} - y$⁩)؛ گرادیان‌ها را در دیکشنری `grads` بریزد؛
- `params`: دیکشنری پارامترها.

**راهنمایی گرادیان خروجی:** اگر ⁦$L$⁩ cross-entropy با softmax باشد، ⁦$\frac{\partial L}{\partial a^{(L)}} = \frac{1}{n}(\mathrm{softmax}(a^{(L)}) - Y_{onehot})$⁩.

</div>

In [ ]:
class MLP:
    def __init__(self, hidden=(64, 64), seed=SEED):
        # TODO: مقداردهی He برای وزن‌ها
        pass
    def forward(self, X):
        # TODO
        pass
    def backward(self, X, Y_onehot):
        # TODO: پر کردن self.grads
        pass


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

</div>

In [ ]:
# بررسی گرادیان با تفاضل متناهی — فقط اجرا کنید
mlp = MLP(hidden=(8, 8), seed=SEED)
Xs_ = X4_tr[:20]; Ys_ = to_onehot(y4_tr[:20])
mlp.forward(Xs_); mlp.backward(Xs_, Ys_)
logits = mlp.forward(Xs_)

def loss_of(mlp, X, Y):
    P = mlp.softmax(mlp.forward(X))
    return -np.mean(np.log((P * Y).sum(1) + 1e-300))

eps = 1e-6
# بررسی dW0[0,0] و db1[1]
i, j = 0, 0
old = mlp.W[0][i, j]
mlp.W[0][i, j] = old + eps; lp = loss_of(mlp, Xs_, Ys_)
mlp.W[0][i, j] = old - eps; lm = loss_of(mlp, Xs_, Ys_)
mlp.W[0][i, j] = old
num = (lp - lm) / (2 * eps)
assert abs(num - mlp.grads['dW0'][i, j]) < 1e-5, f"dW0 mismatch: {num} vs {mlp.grads['dW0'][i,j]}"
old = mlp.b[1][1]
mlp.b[1][1] = old + eps; lp = loss_of(mlp, Xs_, Ys_)
mlp.b[1][1] = old - eps; lm = loss_of(mlp, Xs_, Ys_)
mlp.b[1][1] = old
num = (lp - lm) / (2 * eps)
assert abs(num - mlp.grads['db1'][1]) < 1e-5, f"db1 mismatch: {num} vs {mlp.grads['db1'][1]}"
print("گرادیان‌های MLP با تفاضل متناهی سازگارند ✓")

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۴-ب) (۸ نمره)** بهینه‌ساز Adam را پیاده کنید: کلاس `Adam` با به‌روزرسانی

<div dir="ltr">

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2
$$

</div>
<div dir="ltr">

$$
\hat m = \frac{m_t}{1-\beta_1^t}, \quad \hat v = \frac{v_t}{1-\beta_2^t}, \quad \theta \leftarrow \theta - \eta \frac{\hat m}{\sqrt{\hat v} + \epsilon}
$$

</div>

با پیش‌فرض‌های ⁦$\beta_1=0.9$⁩، ⁦$\beta_2=0.999$⁩، ⁦$\epsilon=10^{-8}$⁩. سپس حلقهٔ آموزش `train_mlp(optimizer='sgd' یا 'adam', lr, epochs)` بنویسید که **full-batch** آموزش می‌دهد و تاریخچهٔ هزینه را برمی‌گرداند. هر دو بهینه‌ساز را با نرخ یادگیری یکسان ⁦$0.01$⁩ آموزش دهید، منحنی هزینه‌ها را در یک نمودار مقایسه کنید و دقت آزمون هر دو را گزارش کنید.

</div>

In [ ]:
class Adam:
    def __init__(self, params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        # TODO
        pass
    def step(self, grads):
        # TODO: به‌روزرسانی همهٔ پارامترها
        pass

def train_mlp(X, y, optimizer='adam', lr=0.01, epochs=300, seed=SEED):
    # TODO: آموزش full-batch؛ خروجی (mlp, history)
    pass


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۴-ج) (۷ نمره)** مرز تصمیم: مرز تصمیم مدل Adam را روی شبکهٔ نقاط ⁦$[-3.5, 3.5]^2$⁩ با colormap رسم کنید (احتمال کلاس ۱). سپس در یک سلول markdown توضیح دهید: (۱) چرا مدل خطی (مثل مسئلهٔ ۱) روی این داده شکست می‌خورد؛ (۲) تفاوت رفتار همگرایی Adam و SGD که در نمودار دیدید و نقش گام‌های تطبیقی.

</div>

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

## مسئلهٔ ۵ — شبکهٔ کانولوشنی با PyTorch (۲۵ نمره)

یک مجموعه‌دادهٔ تصویری **سنتزی** می‌سازیم: تصاویر سیاه‌وسفید ⁦$28\times28$⁩ که در آن‌ها یا «دایره» یا «مربع» یا «مثلث» (کلاس‌ها) در مکان و اندازهٔ تصادفی رسم شده است. سپس یک CNN کوچک برای طبقه‌بندی شکل آموزش می‌دهیم — همه‌چیز روی CPU و در چند دقیقه.

</div>

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

</div>

In [23]:
# ساخت داده — فقط اجرا کنید، تغییر ندهید
import math
def draw_shape(img, cls, rng):
    cx, cy = rng.integers(8, 20), rng.integers(8, 20)
    r = rng.integers(4, 8)
    if cls == 0:      # دایره
        for t in np.linspace(0, 2*np.pi, 200):
            x, y = int(cx + r*np.cos(t)), int(cy + r*np.sin(t))
            if 0 <= x < 28 and 0 <= y < 28: img[y, x] = 1
    elif cls == 1:    # مربع
        x0, y0 = cx - r, cy - r
        for s in np.linspace(-r, r, 4*r+1):
            for (x, y) in [(x0+s, y0), (x0+s, y0+2*r), (x0, y0+s), (x0+2*r, y0+s)]:
                if 0 <= x < 28 and 0 <= y < 28: img[int(y), int(x)] = 1
    else:             # مثلث
        pts = [(cx, cy - r), (cx - r, cy + r), (cx + r, cy + r)]
        for (x1, y1), (x2, y2) in [(pts[0], pts[1]), (pts[1], pts[2]), (pts[2], pts[0])]:
            for s in np.linspace(0, 1, 4*r+1):
                x, y = int(x1 + s*(x2-x1)), int(y1 + s*(y2-y1))
                if 0 <= x < 28 and 0 <= y < 28: img[y, x] = 1

def make_shapes_dataset(n_per, seed):
    rng = np.random.default_rng(seed)
    X = np.zeros((3*n_per, 1, 28, 28), dtype=np.float32)
    y = np.zeros(3*n_per, dtype=np.int64)
    for i in range(3*n_per):
        cls = i % 3
        draw_shape(X[i, 0], cls, rng)
        y[i] = cls
    return X, y

Xc_tr, yc_tr = make_shapes_dataset(400, SEED)
Xc_te, yc_te = make_shapes_dataset(100, SEED + 50)
fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for c, name in enumerate(['دایره', 'مربع', 'مثلث']):
    axes[c].imshow(Xc_tr[yc_tr == c][0, 0], cmap='gray'); axes[c].set_title(name); axes[c].axis('off')
plt.show()
print("ابعاد دادهٔ آموزش:", Xc_tr.shape, "| آزمون:", Xc_te.shape)

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۵-الف) (۵ نمره)** پاسخ دهید (در سلول markdown، با محاسبهٔ دستی):
1. یک لایهٔ `nn.Conv2d(1, 16, kernel_size=3, padding=1)` چند پارامتر (وزن + بایاس) دارد؟
2. اگر ورودی ⁦$1\times28\times28$⁩ از سه لایهٔ کانولوشنی (کرنل ⁦$3\times3$⁩، padding=1، stride=1) با ۱۶، ۳۲ و ۳۲ کانال و سپس `AdaptiveAvgPool2d(1)` عبور کند، ابعاد تانسور در هر مرحله چیست؟
3. چرا برای این مسئله CNN نسبت به MLP با ورودی مسطح (⁦$784$⁩) مناسب‌تر است؟ (دو دلیل بنویسید.)

</div>

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۵-ب) (۱۰ نمره)** کلاس `ShapeCNN` را بنویسید با معماری: سه بلوک [`Conv2d(3×3, pad=1)` + `BatchNorm2d` + `ReLU` + `MaxPool2d(2)`] با کانال‌های ۱۶، ۳۲، ۶۴، سپس `AdaptiveAvgPool2d(1)` و یک لایهٔ خطی به ۳ کلاس. تابع `count_params(model)` را هم بنویسید که تعداد کل پارامترهای آموزش‌پذیر را برگرداند و برای مدل خودتان چاپ کنید.

</div>

In [ ]:
class ShapeCNN(nn.Module):
    def __init__(self):
        # TODO
        pass
    def forward(self, x):
        # TODO
        pass

def count_params(model):
    # TODO
    pass


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۵-ج) (۱۰ نمره)** حلقهٔ آموزش بنویسید: `CrossEntropyLoss`، بهینه‌ساز `Adam` با lr=⁦$10^{-3}$⁩، ۳ epoch، اندازهٔ batch=64. در هر epoch دقت آموزش و در پایان دقت آزمون را گزارش کنید (توقع: دقت آزمون بالای ⁦$95\%$⁩). سپس **confusion matrix** ⁦$3\times3$⁩ را برای دادهٔ آزمون به‌صورت عددی چاپ یا رسم کنید و بگویید مدل در تشخیص کدام دو شکل بیشترین اشتباه را داشته است.

</div>

In [ ]:
def train_cnn(model, X, y, X_te, y_te, epochs=3, bs=64, lr=1e-3):
    # TODO: آموزش mini-batch؛ گزارش دقت هر epoch و دقت آزمون نهایی + confusion matrix
    pass

train_cnn(model_cnn, Xc_tr, yc_tr, Xc_te, yc_te)


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

## مسئلهٔ ۶ — attention و Transformer کوچک (۲۵ نمره)

در این مسئله اول هستهٔ اصلی Transformer — یعنی scaled dot-product attention — را از صفر پیاده می‌کنیم و بعد یک Transformer کوچک روی یک وظیفهٔ الگوریتمی آموزش می‌دهیم.

**وظیفه:** دنباله‌هایی از ارقام ⁦$0..9$⁩ به طول ⁦$12$⁩ داده می‌شود؛ مدل باید تعداد ارقامی که **بزرگ‌تر یا مساوی ۵** هستند را بشمارد و به‌صورت یکی از ۱۳ کلاس (⁦$0..12$⁩) خروجی دهد. یعنی برچسب = ⁦$\big|\{i : x_i \ge 5\}\big|$⁩. این وظیفه یک selective counting است: مدل باید یاد بگیرد با توجه، فقط توکن‌های «بزرگ» را جمع بزند و از بقیه صرف‌نظر کند.

</div>

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۶-الف) (۸ نمره)** تابع `scaled_dot_product_attention(Q, K, V)` را بنویسید:

<div dir="ltr">

$$
\mathrm{Attn}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

</div>

ورودی‌ها شکل ⁦$(B, H, T, d_k)$⁩ دارند. تابع باید خروجی و **وزن‌های توجه** را برگرداند. سپس با یک آزمایش کوچک صحت شکل‌ها را بررسی کنید و برای یک ⁦$Q,K,V$⁩ تصادفی نشان دهید سطرهای ماتریس توجه جمعشان ۱ است.

</div>

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    # TODO: خروجی (output, attn_weights)
    pass


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۶-ب) (۸ نمره)** کلاس `MultiHeadAttention` را بنویسید: پروژه‌های ⁦$W_Q, W_K, W_V, W_O$⁩ (به‌صورت `nn.Linear` بدون بایاس)، تقسیم به ⁦$H$⁩ سر، استفاده از تابع بخش الف، و concatenation of heads. سپس لایهٔ `TransformerBlock` را بسازید: multi-head attention + residual connection + LayerNorm (pre-norm) + MLP دولایه با فعال‌سازی GELU + residual connection + LayerNorm:

<div dir="ltr">

$$
x \leftarrow x + \mathrm{MHA}(\mathrm{LN}(x)), \qquad x \leftarrow x + \mathrm{MLP}(\mathrm{LN}(x))
$$

</div>

</div>

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        # TODO
        pass
    def forward(self, x):
        # TODO: خروجی (out, attn_weights)
        pass

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        # TODO
        pass
    def forward(self, x):
        # TODO (pre-norm)
        pass


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۶-ج) (۹ نمره)** مدل `SumTransformer` را بسازید: embedding توکن (⁦$10+1$⁩ توکن: ارقام + query token)، ⁦$2$⁩ بلوک `TransformerBlock` با ⁦$d_{model}=64$⁩، ⁦$H=4$⁩، ⁦$d_{ff}=128$⁩، و یک linear head به **۱۳ کلاس** که فقط از موقعیت توکن [CLS] (ابتدای دنباله) خروجی می‌گیرد.

روتین آموزش: دادهٔ ⁦$N=5000$⁩ دنبالهٔ تصادفی با برچسب تعداد ارقام ⁦$\ge 5$⁩؛ تقسیم ۹۰/۱۰ آموزش/آزمون؛ `Adam` با lr=⁦$3\times10^{-4}$⁩، ۱۰ epoch، batch=128. دقت طبقه‌بندی exact-match را گزارش کنید (توقع: به‌مراتب بالاتر از حد تصادفی ⁦$\approx 8\%$⁩؛ مدل موفق به بالای ⁦$90\%$⁩ می‌رسد) و همچنین دقت «تا خطای یک» (نسبت پیش‌بینی‌هایی که حداکثر یکی با برچسب فرق دارند) را هم گزارش کنید. منحنی دقت آموزش را رسم کنید.

**نکتهٔ پیاده‌سازی:** دنبالهٔ ورودی = [توکن [CLS]] + ارقام؛ برچسب = تعداد ارقام ⁦$\ge5$⁩؛ loss فقط روی logits موقعیت ۰ (توکن [CLS]) محاسبه شود. چون وظیفه به مکان توکن‌ها وابسته نیست (فقط محتوایشان مهم است)، از positional embedding صرف‌نظر کنید.

</div>

In [ ]:
class SumTransformer(nn.Module):
    def __init__(self, vocab=11, d_model=64, n_heads=4, n_blocks=2, d_ff=128, n_classes=13):
        # TODO: 10 رقم + 1 توکن ویژه
        pass
    def forward(self, x):
        # TODO: logits از موقعیت توکن ویژه
        pass

def make_count_data(n, seq_len, seed):
    # TODO: دنباله‌ها و برچسب تعداد ارقام >= 5
    pass

# TODO: آموزش + گزارش دقت (exact و ±1) + رسم منحنی دقت


<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

**۶-د) (بخش اختیاری؛ نمرهٔ اضافه تا ۵)** وزن‌های توجه سر اولِ بلوک اول را برای یک نمونهٔ آزمون به‌صورت heatmap رسم کنید و تفسیر کنید: مدل به کدام توکن‌ها (مثلاً توکن‌های با مقدار بزرگ) توجه بیشتری دارد؟

</div>

<div dir="rtl" lang="fa" style="font-family: 'Vazirmatn', sans-serif; line-height: 1.9; direction: rtl; text-align: right;">

---

<div dir="rtl" align="center">

**پایان آزمون عملی — موفق باشید**

</div>

</div>